<a href="https://colab.research.google.com/github/yana-moshnikova/python-ai-moshnikova-yana/blob/main/notebooks/week2b_read_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Week 2: Data Analysis — Чтение и проверка данных

**Цель**: Научиться читать CSV-файлы из репозитория GitHub в Google Colab и выполнять базовую проверку данных с помощью pandas.

**Данные:**
- [`art_museum.csv`](https://github.com/yana-moshnikova/python-ai-moshnikova-yana/blob/main/data/art_museum.csv) — название музея, год основания, страна, посетители, площадь, подписчики в соцсетях.
- [`art_museum_visitors.csv`](https://github.com/yana-moshnikova/python-ai-moshnikova-yana/blob/main/data/art_museum_visitors.csv) — данные о посетителях художественных музеев с указанием года (квалификатор даты).


**Что мы делаем:**
1. Клонируем репозиторий GitHub в Colab
2. Читаем CSV-файл в pandas DataFrame
3. Очищаем и переименовываем столбцы
4. Смотрим структуру данных и делаем быструю валидацию

## 🐱 [1] Клонируем репозиторий курса в Colab

In [11]:
# 🐱 Шаг 1. Клонируем репозиторий курса в Colab

import os

repo = "python-ai-moshnikova-yana"
repo_path = f"/content/{repo}"  # абсолютный путь — не зависит от cwd

if not os.path.exists(repo_path):
    !git clone -q https://github.com/yana-moshnikova/python-ai-moshnikova-yana.git

if os.getcwd() != repo_path:
    %cd {repo_path}

print("✅ Репозиторий готов, теперь мы работаем внутри папки", repo)

✅ Репозиторий готов, теперь мы работаем внутри папки python-ai-moshnikova-yana


## 📥 [2A] Простое чтение CSV-файла в pandas

Сначала просто прочитаем CSV-файлы в объекты `DataFrame`, без каких‑либо изменений.

После этого мы узнаем, сколько строк загружено в каждый датасет.

In [12]:
# 🐱 Шаг 2A. Чтение двух CSV-файлов в pandas

import pandas as pd
import os

# 🔍 Проверяем текущую директорию (для отладки)
print(f"📁 Текущая папка: {os.getcwd()}")

# Загружаем основной файл с информацией о музеях
# Путь относительный: data/art_museum.csv внутри папки репозитория
df_museum = pd.read_csv("data/art_museum.csv")
print(f"✅ Загружено {len(df_museum)} строк из data/art_museum.csv")

# Загружаем файл с посещаемостью по годам (квалификатор P585)
df_visitors = pd.read_csv("data/art_museum_visitors.csv")
print(f"✅ Загружено {len(df_visitors)} строк из data/art_museum_visitors.csv")

print(f"\n📋 Столбцы в df_museum: {df_museum.columns.tolist()}")
print(f"📋 Столбцы в df_visitors: {df_visitors.columns.tolist()}")

# 🔍 Быстрая проверка на дубликаты столбцов
for name, df in [("df_museum", df_museum), ("df_visitors", df_visitors)]:
    if df.columns.duplicated().any():
        print(f"⚠️ {name}: найдены дубликаты имён столбцов!")

📁 Текущая папка: /content/python-ai-moshnikova-yana
✅ Загружено 8677 строк из data/art_museum.csv


FileNotFoundError: [Errno 2] No such file or directory: 'data/art_museum_visitors.csv'

## 🧹 [2B] Очистка, переименование и объединение данных

### Что мы делаем:

1️⃣ **Переименование**: убираем суффикс `Label` у столбцов (`museumLabel → museum`)

2️⃣ **Приведение типов**: числовые поля (`inceptionYear`, `area`, `visitors` и др.) конвертируем в `int`, некорректные значения → `NaN`

3️⃣ **Проверка заполненности OPTIONAL-полей** (ДО `fillna(0)`):
   - Считаем долю непустых значений через `notna().mean()`
   - Это важно: после замены `NaN → 0` невозможно отличить «нет данных» от «действительно ноль»

4️⃣ **Объединение таблиц** через `pd.merge(..., on="URL")`:
   - `how="left"`: сохраняем все музеи, даже если нет данных о посетителях
   - Результат: один музей может иметь несколько строк (по одной на каждый год с данными)

5️⃣ **Заполнение пропусков**: только после всех проверок заменяем `NaN → 0`

6️⃣ **(Опционально) Агрегация**: если нужна одна строка на музей — раскомментируйте блок в коде

> 💡 **Зачем так:** Временная гранулярность позволяет строить графики динамики посещаемости 📈. Если нужен «плоский» датасет для топов — используйте агрегацию.

> ⚠️ **Важно:** Все последующие шаги работают с переменной `df_museum`, которая после этого шага содержит объединённые данные.

In [4]:
# 🧹 Шаг 2B. Очистка, переименование и объединение данных

# 🔍 1. Очистка и переименование df_museum
if "museumLabel" in df_museum.columns:
    df_museum = df_museum.rename(columns={
        "museumLabel": "museum",
        "countryLabel": "country",
    })
    print("✅ df_museum: переименованы Label-столбцы")

# 🔍 2. Очистка и переименование df_visitors
if "museumLabel" in df_visitors.columns:
    df_visitors = df_visitors.rename(columns={
        "museumLabel": "museum",
        "countryLabel": "country",
    })
    print("✅ df_visitors: переименованы Label-столбцы")

# 🔍 3. Убираем дубликаты имён столбцов (если есть)
for df in [df_museum, df_visitors]:
    if df.columns.duplicated().any():
        df = df.loc[:, ~df.columns.duplicated()]

# 🔢 4. Приведение числовых столбцов к числовому типу (без fillna пока!)
numeric_museum = ["inceptionYear", "area", "socialFollowers"]
for col in numeric_museum:
    if col in df_museum.columns:
        df_museum[col] = pd.to_numeric(df_museum[col], errors="coerce")

numeric_visitors = ["visitors", "year"]
for col in numeric_visitors:
    if col in df_visitors.columns:
        df_visitors[col] = pd.to_numeric(df_visitors[col], errors="coerce")

print("✅ Числовые столбцы приведены к числовому типу (NaN сохранён)")

# 📊 5. Считаем заполненность OPTIONAL-полей ДО fillna(0)
print(f"\n📈 Заполненность OPTIONAL-полей (ДО замены пропусков на 0):")
for col in ["area", "socialFollowers"]:
    if col in df_museum.columns:
        completeness = df_museum[col].notna().mean()
        print(f"   • {col} (музеи): {completeness:.1%} ({df_museum[col].notna().sum()} из {len(df_museum)})")
if "visitors" in df_visitors.columns:
    completeness = df_visitors["visitors"].notna().mean()
    print(f"   • visitors (по годам): {completeness:.1%} ({df_visitors['visitors'].notna().sum()} из {len(df_visitors)})")

# 🔗 6. Объединяем таблицы по URL
if "URL" in df_museum.columns and "URL" in df_visitors.columns:
    df_combined = pd.merge(
        df_museum,
        df_visitors,
        on="URL",
        how="left",
        suffixes=("", "_vis")
    )
    print(f"\n✅ Объединено: {len(df_combined)} записей после merge по URL")
    print(f"   - Уникальных музеев: {df_combined['museum'].nunique()}")
    print(f"   - Уникальных лет с данными: {df_combined['year'].dropna().nunique()}")
else:
    print("⚠️ Столбец 'URL' не найден, пропускаем объединение")
    df_combined = df_museum.copy()

# 🧹 7. Приводим типы после merge и заполняем пропуски нулями
for col in ["inceptionYear", "area", "socialFollowers", "visitors", "year"]:
    if col in df_combined.columns and df_combined[col].isna().any():
        df_combined[col] = df_combined[col].fillna(0).astype(int)

# 📦 8. (Опционально) Агрегация: одна строка на музей
# Раскомментируйте, если в дальнейших ячейках нужна агрегированная таблица
"""
print(f"\n📊 Агрегация: одна строка на музей...")
df_combined = df_combined.groupby(["museum", "country"], as_index=False).agg({
    "URL": "first",
    "inceptionYear": lambda x: x[x > 0].min() if (x > 0).any() else 0,
    "area": "max",
    "socialFollowers": "max",
    "visitors": "max",
    "year": lambda x: x.max() if x.max() > 0 else 0
})
print(f"✅ После агрегации: {len(df_combined)} уникальных музеев")
"""

# 🎯 9. Итоговая переменная
df_museum = df_combined

print("\n✅ Данные готовы к анализу")
print(f"\n📋 Итоговая структура df_museum:")
print(f"   - Строк: {len(df_museum)}")
print(f"   - Столбцов: {len(df_museum.columns)}")
print(f"   - Названия столбцов: {df_museum.columns.tolist()}")



✅ df_museum уже загружен в память

🔍 Диагностика df_museum:
   - Столбцы: ['museum', 'museum', 'inceptionYear', 'country', 'visitors', 'area', 'socialFollowers']
   - Есть дубликаты имён столбцов: True
⚠️ Найдены дубликаты имён столбцов! Убираем...
⏭️ df_museum уже очищен, пропускаем переименование

📊 До группировки: 8677 записей
✅ После группировки: 4030 уникальных музеев

✅ Данные готовы к анализу

📋 Итоговая структура df_museum:
   - Строк: 4030
   - Столбцов: 6
   - Названия столбцов: ['museum', 'country', 'inceptionYear', 'visitors', 'area', 'socialFollowers']


## 🔍 [3] Обзор данных: структура и первые строки

Сделаем короткий обзор DataFrame с данными о музеях:

- посмотрим размер таблицы (`shape`);
- выведем список столбцов;
- посмотрим первые несколько строк;
- дополнительно посчитаем базовую статистику по числовым столбцам (`visitors`, `area`, `socialFollowers`).

Для удобства напишем маленькую функцию `show_info(df, name)`, чтобы получить сводную информацию о датасете в одном месте.

In [ ]:
def show_info(df, name, n=5):
    """Краткий обзор DataFrame: имя, размер, список столбцов и первые строки."""
    print(f"\n📊 {name}")
    print("Размер:", df.shape)
    print("Столбцы:", ", ".join(df.columns))
    print("\nПервые строки:")
    print(df.head(n))
    print("\n📈 Базовая статистика по числовым столбцам:")
    print(df.describe())

# 🔍 Шаг 3. Обзор данных

show_info(df_museum, "Данные о музеях (df_museum)")


📊 Данные о музеях (df_museum)
Размер: (3228, 6)
Столбцы: museum, inceptionYear, country, visitors, area, socialFollowers

Первые строки:
                  museum  inceptionYear  country  visitors   area  \
0  Музей театра Ла Скала           1913   Италия    260478    600   
1  Музей театра Ла Скала           1913   Италия    260478    850   
2                Версаль           1661  Франция   6746198  67000   
3                Версаль           1661  Франция   6746198  67000   
4                Версаль           1661  Франция   2593260  67000   

   socialFollowers  
0           309856  
1           309856  
2           100000  
3           497943  
4           100000  

📈 Базовая статистика по числовым столбцам:
       inceptionYear      visitors           area  socialFollowers
count    3228.000000  3.228000e+03    3228.000000     3.228000e+03
mean     1871.340458  1.081145e+05    4586.326828     9.588588e+03
std       271.250149  6.000339e+05   23704.857020     1.206914e+05
min      

## ✅ [4] Быстрая проверка и валидация данных

Здесь мы посмотрим:

- сколько **уникальных** музеев и стран есть в данных;
- **какие страны встречаются чаще всего** (Топ‑5 по числу записей);
- **диапазон лет основания** музеев (`inceptionYear`);
- **базовую статистику** по числовым показателям: посещаемость (`visitors`), площадь (`area`), подписчики в соцсетях (`socialFollowers`).

Функция `value_counts()`:
- считает, сколько раз каждое значение встречается в столбце;
- сортирует результаты по убыванию.

Метод `.head()` берёт первые N строк, поэтому
`df_museum["country"].value_counts().head()` даёт **Топ‑5 стран по числу записей**.

> 💡 **Зачем это нужно:** быстрая валидация помогает выявить аномалии — например, музеи с нулевой площадью, отрицательным числом посетителей или дубликаты названий.

In [ ]:
# ✅ Шаг 4. Быстрая проверка и валидация данных

print("🔍 Быстрая проверка данных")

# Датасет: музеи (df_museum уже сгруппирован в Шаге 2B)
print("\n📊 Датасет: Музеи (df_museum)")
print("Уникальных музеев:", df_museum["museum"].nunique())
print("Уникальных стран:", df_museum["country"].nunique())

print("\n📅 Диапазон лет основания:")
print("Мин:", df_museum["inceptionYear"].min(), "— Макс:", df_museum["inceptionYear"].max())

print("\n💰 Диапазон посещаемости (человек):")
print("Мин:", df_museum["visitors"].min(), "— Макс:", df_museum["visitors"].max())

print("\n🏛️ Диапазон площади (м²):")
print("Мин:", df_museum["area"].min(), "— Макс:", df_museum["area"].max())

print("\n📱 Диапазон подписчиков в соцсетях:")
print("Мин:", df_museum["socialFollowers"].min(), "— Макс:", df_museum["socialFollowers"].max())

# Топ-5 стран по числу записей
print("\nТоп-5 стран по числу записей:")
print(df_museum["country"].value_counts().head())

# 🔥 Интересная статистика

print("\n⭐ Топ-5 музеев по количеству подписчиков в соцсетях:")
top_followers = df_museum.nlargest(5, "socialFollowers")[["museum", "country", "socialFollowers"]]
print(top_followers.to_string(index=False))

print("\n🏛️ Топ-5 музеев по площади:")
top_area = df_museum.nlargest(5, "area")[["museum", "country", "area"]]
print(top_area.to_string(index=False))

print("\n🕰️ Топ-5 самых старых музеев (только с достоверным годом основания):")
# Фильтруем: год основания должен быть реалистичным (>= 1000 и <= текущий год)
current_year = 2026
df_with_valid_year = df_museum[
    df_museum["inceptionYear"].notna() &
    (df_museum["inceptionYear"] >= 1000) &
    (df_museum["inceptionYear"] <= current_year)
]

if len(df_with_valid_year) > 0:
    top_oldest = df_with_valid_year.nsmallest(5, "inceptionYear")[["museum", "country", "inceptionYear"]]
    print(top_oldest.to_string(index=False))
    print(f"\n💡 Всего музеев с достоверным годом основания: {len(df_with_valid_year)} из {len(df_museum)}")
else:
    print("⚠️ Нет данных с достоверным годом основания для построения рейтинга")

print("\n📈 Статистика по числовым показателям:")
print(df_museum[["visitors", "area", "socialFollowers"]].describe())

print("\n🔎 Проверка на пропуски (NaN):")
print(df_museum.isnull().sum())

# 🔍 Бонус: поиск подозрительных значений в годе основания
suspicious = df_museum[
    (df_museum["inceptionYear"] < 1000) |
    (df_museum["inceptionYear"] > current_year) |
    (df_museum["inceptionYear"] == 0)
]["inceptionYear"].unique()

if len(suspicious) > 0:
    print("\n⚠️ Подозрительные значения в inceptionYear:", sorted(suspicious))
else:
    print("\n✅ Все годы основания выглядят реалистично")

print("\n✅ Данные прошли базовую валидацию")

🔍 Быстрая проверка данных

📊 Датасет: Музеи (df_museum)
Уникальных музеев: 529
Уникальных стран: 30

📅 Диапазон лет основания:
Мин: 0 — Макс: 2025

💰 Диапазон посещаемости (человек):
Мин: 0 — Макс: 8132518

🏛️ Диапазон площади (м²):
Мин: 0 — Макс: 523087

📱 Диапазон подписчиков в соцсетях:
Мин: 0 — Макс: 4704971

Топ-5 стран по числу записей:
country
Италия     444
Испания     16
Франция     12
США          9
Камерун      6
Name: count, dtype: int64

⭐ Топ-5 музеев по количеству подписчиков в соцсетях:
                    museum        country  socialFollowers
               Тейт Модерн Великобритания          4704971
Музей Соломона Гуггенхайма            США          3387252
                      Лувр        Франция          1579945
    Museo Nacional de Arte        Мексика          1408505
                     Прадо        Испания          1313537

🏛️ Топ-5 музеев по площади:
                                  museum                       country   area
                        Пинакот

## 📝 Summary

**Что мы сделали в этом ноутбуке (Week 2):**

- ✅ Клонировали репозиторий GitHub в Colab
- ✅ Прочитали CSV-файл `data/art_museum.csv`
- ✅ Переименовали столбцы с суффиксом `Label` (`museumLabel → museum`, `countryLabel → country`)
- ✅ Привели числовые столбцы к типу `int` (`inceptionYear`, `visitors`, `area`, `socialFollowers`)
- ✅ Проверили структуру данных (размер, столбцы, первые строки)
- ✅ Выполнили быструю валидацию:
  - количество уникальных музеев и стран
  - диапазон лет основания музеев
  - топ стран по числу записей
  - статистику по посещаемости, площади и подписчикам в соцсетях
  - проверку на пропущенные значения (NaN)

Теперь у нас есть **аккуратная, проверенная таблица** с данными о музеях, с которой удобно работать дальше.

В отдельном ноутбуке для следующей недели мы будем использовать **те же данные** для:
- более сложного анализа (группировки по странам, фильтрация по годам),
- и построения визуализаций (графики посещаемости, диаграммы распределения музеев). 🎨